# ROR raw EDA

## Things i need to find out

* Geolocation, country, citz
 
## Sanitization

We can use Python UDF's for that

```
con.create_function("parse_names", parse_names_and_identifiers)
con.execute("""
    UPDATE organization 
    SET legalName = parse_names(legalName)
""")
```

In [1]:
import sys
sys.path.insert(0, '/home/lu72hip/DIGICHer/dh_pipeline/src')

import pandas as pd
from pathlib import Path
from lib.database.duck.create_connection import create_duck_connection
from utils.config.config_loader import get_query_config

config = get_query_config()['ror_dump']
DB = Path(config['path_duck'])

con = create_duck_connection(str(DB))
con.execute("SET memory_limit='16GB'")
con.execute("SET threads=8")
print(f'Connected to {DB}')

pd.set_option("display.max_rows", 50) # Show more rows  # or None for unlimited
pd.set_option("display.max_columns", None) # Show more columns  # None = show all
pd.set_option("display.max_colwidth", None) # Don't truncate column content  # None = full content
pd.set_option("display.width", None) # Wider display  # Auto-detect terminal width

/home/lu72hip/DIGICHer/dh_pipeline/config/config_queries.json {'arxiv': {'checkpoint': 'submittedDateTo', 'queries': [{'query': '(all:cultural+OR+all:heritage+OR+all:digital+OR+all:humanities)', 'download_attachments': True, 'checkpoint_start': '1990-01-01-00-00', 'checkpoint_range': '6'}, {'query': 'all:computing+AND+(all:humanities+OR+all:heritage)', 'download_attachments': True, 'checkpoint_start': '1990-01-01-00-00', 'checkpoint_range': '6'}]}, 'cordis': {'checkpoint': 'startDate', 'queries': [{'query': 'cultural OR heritage OR digital OR humanities', 'download_attachments': True, 'checkpoint_start': '1985-01-01', 'checkpoint_range': '1'}, {'query': "contenttype='project' AND '*'", 'download_attachments': False, 'checkpoint_start': '1985-01-01', 'checkpoint_range': '1'}]}, 'coreac': {'checkpoint': 'publishedDate', 'queries': [{'query': '((computing AND cultural) OR (computing AND heritage))', 'download_attachments': False, 'checkpoint_start': '1990-01-01', 'checkpoint_range': '10'}

## Stats

In [2]:
q = """ 
SELECT
    table_name,
    estimated_size as cnt
FROM duckdb_tables()
WHERE schema_name = 'main'
ORDER BY estimated_size DESC
"""
con.execute(q).df()

,table_name,cnt
0,organizations,122388


## Entity Organization

* geolocation missing
    * For 120k/340k ROR orgs i can get it easily
    * GRID is in ROR right?
    * PIC 70342
    * ISNI 60138
    * Wikidata 57673 -> ROR overlap
    * mag_id 27080
    * FundRef 22750
    * OrgRef 16448
    * RNSR 9157
    * OrgReg 6639
    * For the rest i did have a mapbox script i think, though i have little meta data, just name and country

### Sanitization Organization
* BOM needs to be cleaned in text, eg value.lstrip('\ufeff\u200b\u200c\u200d\ufffe')

In [9]:
### Sample Columns ###
q = """ 
SELECT names FROM organizations
LIMIT 3;
"""
display(con.execute(q).df())

### Seach FSU Jena ###
q = """ 
SELECT unnest(names) FROM organizations
-- WHERE names.value ilike '%jena%'
LIMIT 3;
"""
display(con.execute(q).df())

,names
0,"[{'value': 'RMIT', 'types': ['acronym'], 'lang': None}, {'value': 'RMIT University', 'types': ['ror_display', 'label'], 'lang': 'en'}, {'value': 'Royal Melbourne Institute of Technology University', 'types': ['alias'], 'lang': 'en'}]"
1,"[{'value': 'La Trobe University', 'types': ['ror_display', 'label'], 'lang': 'en'}]"
2,"[{'value': 'CQU', 'types': ['acronym'], 'lang': None}, {'value': 'CQUniversity', 'types': ['alias'], 'lang': 'en'}, {'value': 'Central Queensland University', 'types': ['ror_display', 'label'], 'lang': 'en'}]"


,"unnest(""names"")"
0,"{'value': 'RMIT', 'types': ['acronym'], 'lang': None}"
1,"{'value': 'RMIT University', 'types': ['ror_display', 'label'], 'lang': 'en'}"
2,"{'value': 'Royal Melbourne Institute of Technology University', 'types': ['alias'], 'lang': 'en'}"


In [ ]:
### Duplicate legalName's ###

q = """ 
SELECT legalName, COUNT(*) as cnt
FROM organization 
GROUP BY legalName
HAVING COUNT(*) > 1
ORDER BY cnt DESC"""
# display(con.execute(q).df())

q = """ 
SELECT * FROM organization 
WHERE legalName = 'Ministry of Health'"""
# display(con.execute(q).df())

In [55]:
### Grouped Organization Sources ###

q = """ 
SELECT 
    pid.scheme,
    COUNT(*) as cnt
FROM organization,
UNNEST(pids) as t(pid)
GROUP BY pid.scheme
ORDER BY cnt DESC
"""
display(con.execute(q).df())

,scheme,cnt
0,ROR,124709
1,GRID,109429
2,PIC,70342
3,ISNI,60138
4,Wikidata,57673
5,mag_id,27080
6,FundRef,22750
7,OrgRef,16448
8,RNSR,9157
9,OrgReg,6639


## Entity Project

In [ ]:
q = """ 
SELECT * FROM project
WHERE title != 'unidentified'
--LIMIT 3
"""
display(con.execute(q).df())

In [ ]:
### For each PID scheme (ROR, GRID etc.), how many projects connect to orgs with that scheme?
q = """ 
SELECT 
    op.scheme,
    COUNT(DISTINCT p.id) as projects_count,
    COUNT(DISTINCT o.id) as orgs_count
FROM project p
INNER JOIN relation r ON r.source = p.id AND r.sourceType = 'project'
INNER JOIN organization o ON r.target = o.id AND r.targetType = 'organization'
INNER JOIN organization_pids op ON op.org_id = o.id
WHERE r.relType.type = 'participation'
GROUP BY op.scheme
ORDER BY projects_count DESC;
"""
display(con.execute(q).df())

,scheme,projects_count,orgs_count
0,ROR,899544,17878
1,GRID,898458,16653
2,ISNI,874452,12305
3,Wikidata,872772,11694
4,mag_id,837890,7383
5,OrgRef,835662,5407
6,FundRef,802691,4507
7,PIC,579018,69270
8,RRID,457726,424
9,OrgReg,310012,2774


In [ ]:
### What types of project→org relationships exist?
# Result: No distinction between coordinator / participant
q = """ 
SELECT 
    r.relType.type,
    r.relType.name,
    COUNT(*) as count,
    COUNT(DISTINCT r.source) as unique_projects,
    COUNT(DISTINCT r.target) as unique_orgs
FROM relation r
WHERE r.sourceType = 'project' 
  AND r.targetType = 'organization'
GROUP BY r.relType.type, r.relType.name
ORDER BY count DESC;
"""
display(con.execute(q).df())

,type,name,count,unique_projects,unique_orgs
0,participation,hasParticipant,4568712,3372450,309580


In [64]:
### Question: How many H2020/Horizon projects in OpenAIRE?
# Result:
q = """ 
SELECT 
    CASE 
        WHEN id LIKE 'corda__h2020::%' THEN 'H2020'
        WHEN id LIKE 'corda__%' THEN 'Other CORDIS'
        ELSE 'Non-CORDIS'
    END as source,
    COUNT(*) as project_count,
    COUNT(granted.fundedAmount) as has_funding
FROM project
GROUP BY source
ORDER BY project_count DESC
"""
display(con.execute(q).df())

,source,project_count,has_funding
0,Non-CORDIS,3591397,2018093
1,Other CORDIS,46529,20638
2,H2020,35434,35434


## Entity Publication

In [ ]:
q = """ 
SELECT * FROM publication
LIMIT 10
OFFSET 200000000
"""
display(con.execute(q).df())

In [ ]:
### Grouped Organization Sources ###

q = """ 
SELECT 
    pid.scheme,
    COUNT(*) as cnt
FROM organization,
UNNEST(pids) as t(pid)
GROUP BY pid.scheme
ORDER BY cnt DESC
"""
display(con.execute(q).df())

,scheme,cnt
0,ROR,124709
1,GRID,109429
2,PIC,70342
3,ISNI,60138
4,Wikidata,57673
5,mag_id,27080
6,FundRef,22750
7,OrgRef,16448
8,RNSR,9157
9,OrgReg,6639


## Entity relation

In [ ]:
q = """ 
SELECT count(*) FROM relation
"""
display(con.execute(q).df())

## Impact Metrics

In [65]:
### Question: What impact metrics exist for publications?
# Result:
q = """ 
SELECT 
    COUNT(*) as total_publications,
    COUNT(indicators) as has_indicators,
    COUNT(indicators.citationImpact) as has_citationImpact,
    COUNT(indicators.usageCounts) as has_usageCounts,
    AVG(indicators.citationImpact.citationCount) as avg_citations,
    AVG(indicators.usageCounts.downloads) as avg_downloads,
    AVG(indicators.usageCounts.views) as avg_views
FROM read_json('/vast/lu72hip/data/pile/openaire_2025_12_01_dump/publication/*.json.gz',
    format='newline_delimited', compression='gzip', union_by_name=true)
"""
display(con.execute(q).df())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_publications,has_indicators,has_citationImpact,has_usageCounts,avg_citations,avg_downloads,avg_views
0,205841448,200351379,200347669,3600953,10.627821,57.977084,35.391892
